# ⚡ Optimización de Arquitectura: ¿Por qué convertir a ONNX?

Tras el entrenamiento en PyTorch, el modelo se encuentra en un formato ideal para el aprendizaje, pero no necesariamente para la **operación a gran escala**. La conversión a **ONNX (Open Neural Network Exchange)** es el puente que permite pasar de un experimento de laboratorio a un servicio industrial robusto.

---

### 🚀 Beneficios Estratégicos

La finalidad de este proceso se resume en tres pilares fundamentales:

#### 1. Interoperabilidad (Cero Atadura)
ONNX actúa como un "traductor universal". Al convertir el modelo, dejamos de depender de una librería específica (como PyTorch o TensorFlow). Esto permite que el equipo de **Sentinel** pueda cambiar el motor de inferencia en el futuro sin tener que re-entrenar el modelo desde cero.

#### 2. Maximización del Rendimiento (Latencia)
ONNX optimiza el grafo computacional del modelo. Para el negocio, esto se traduce en:
* **Respuestas más rápidas:** Reducción del tiempo que tarda el bot en entender al cliente.
* **Mayor concurrencia:** Capacidad de procesar más mensajes simultáneos con el mismo hardware.

#### 3. Estándar de Producción en OpenShift AI
Nuestro servidor de modelos (**ModelMesh / OpenVino**) está diseñado para ejecutar archivos ONNX de forma nativa. Al usar este formato, eliminamos la necesidad de instalar dependencias pesadas en producción, resultando en:
* **Contenedores más livianos:** Menor consumo de memoria y almacenamiento.
* **Arranque rápido:** Los pods del servicio inician en segundos ante picos de tráfico.

---

### 📈 Impacto en el Pipeline
En esta etapa, el "cerebro" (pesos del modelo) no cambia, pero su "estructura" se vuelve más eficiente para ser consumida por el microservicio de **FastAPI** y el **Dashboard**.

# 🛠️ Preparación del Entorno de Optimización

Antes de iniciar la conversión del modelo, debemos asegurar que el entorno de ejecución cuente con las herramientasnecesarias.
---

### 📦 Dependencias de Transformación
Utilizamos la librería **Hugging Face Optimum**, una extensión de `transformers` que conecta el entrenamiento con motores de inferencia acelerados.

#### ¿Qué estamos haciendo en este paso?
1. **Instalación de Optimum con ONNX Runtime:** Preparamos el kit de herramientas para traducir el grafo de PyTorch a un formato optimizado para CPU.
2. **Recarga Dinámica del Path:** Dado que estamos instalando librerías "en caliente" durante la ejecución, forzamos a Python a reconocer los nuevos paquetes sin necesidad de reiniciar el kernel de Jupyter.


In [6]:
import sys
import subprocess

# Instalamos de nuevo por si acaso y forzamos la actualización del path
subprocess.check_call([sys.executable, "-m", "pip", "install", "optimum[onnxruntime]"])

# Recargamos el path para que Python vea la nueva librería
import site
from importlib import reload
reload(site)

print("✅ Librería Optimum instalada y cargada en el path.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 4.8 MB/s  0:00:02 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 5.0 MB/s  0:00:00
  Attempting uninstall: huggingface_hub
    Found existing installation: huggingface_hub 1.9.0
    Uninstalling huggingface_hub-1.9.0:
      Successfully uninstalled huggingface_hub-1.9.0
  Attempting uninstall: transformers━━━━━━━━━━━━ 0/2 [huggingface_hub]
    Found existing installation: transformers 5.5.032m0/2 [huggingface_hub]
    Uninstalling transformers-5.5.0:m╺━━━━━━━━━━━━━━━━━━━ 1/2 [transformers]
      Successfully uninstalled transformers-5.5.0━━━━━━━━━━━━━ 1/2 [transformers]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [transformers] [transformers]
✅ Librería Optimum instalada y cargada en el path.


# 🔄 Ejecución de la Exportación a ONNX

En esta sección, utilizamos la clase `ORTModelForSequenceClassification` de la librería **Optimum** para realizar la exportación del grafo. Este proceso toma los pesos entrenados del modelo **BETO** y los traduce a un formato de grafo computacional optimizado.

---

### 🛠️ Pasos del Proceso:

1. **Carga del Modelo PyTorch**: Se leen los archivos de configuración (`config.json`) y los pesos binarios (`pytorch_model.bin`) generados en el paso de entrenamiento.
2. **Generación del Grafo ONNX**: Al usar el parámetro `export=True`, la librería rastrea (traces) el flujo de datos a través de las capas de BERT y genera el archivo `model.onnx`.
3. **Persistencia de Artefactos**:
    * **`model.onnx`**: El modelo optimizado.
    * **Archivos del Tokenizer**: Es fundamental que el tokenizer viaje con el modelo para que la codificación de palabras sea idéntica en producción.

> **💡 Importante**: Una vez finalizado este proceso, el modelo resultante es totalmente independiente de PyTorch y está listo para ser servido por motores de inferencia de alto rendimiento como **OpenVino** o **ONNX Runtime**.

In [7]:
from optimum.onnxruntime import ORTModelForSequenceClassification
from transformers import AutoTokenizer
import os

model_pytorch_path = "./modelo_t-moviles_pytorch"
model_onnx_path = "./modelo_t-moviles_onnx"


print("Iniciando conversión a ONNX (esto puede tardar un minuto)...")
try:
    tokenizer = AutoTokenizer.from_pretrained(model_pytorch_path)
    ort_model = ORTModelForSequenceClassification.from_pretrained(
        model_pytorch_path, 
        export=True
    )

    if not os.path.exists(model_onnx_path):
        os.makedirs(model_onnx_path)
        
    ort_model.save_pretrained(model_onnx_path)
    tokenizer.save_pretrained(model_onnx_path)
    
    print(f"🚀 ¡Éxito! Modelo ONNX listo en: {model_onnx_path}")
    print(f"Archivos generados: {os.listdir(model_onnx_path)}")

except Exception as e:
    print(f"❌ Error en la exportación: {e}")
    print("\n💡 Tip: Verificá que la carpeta './modelo_movistar_pytorch' contenga el archivo 'config.json' y 'pytorch_model.bin'")

Iniciando conversión a ONNX (esto puede tardar un minuto)...
🚀 ¡Éxito! Modelo ONNX listo en: ./modelo_t-moviles_onnx
Archivos generados: ['config.json', 'tokenizer.json', 'model.onnx', 'vocab.txt', 'special_tokens_map.json', 'tokenizer_config.json']
